# 第2章：Tokenizer 与数据工程

## 本章目标
- 理解 BPE (Byte Pair Encoding) 的原理和实现
- 使用 tiktoken 进行 GPT-2 BPE tokenization
- 从零手写一个 mini BPE tokenizer
- 构建训练数据的完整 pipeline

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install tiktoken torch matplotlib
    !wget -q -O input.txt https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
else:
    print("本地环境运行，请确保已按 intro.md 配置好环境")

## Tokenization 速览

BPE 的核心思想：从字符级别开始，反复合并最高频的 token pair，直到达到目标词表大小。

GPT-2 的 BPE：基于 byte-level，词表大小 50,257。
tiktoken 是 OpenAI 开源的高性能 BPE 实现（Rust 后端）。

参考：[BPE Algorithm](https://arxiv.org/abs/1508.07909), [SentencePiece](https://arxiv.org/abs/1808.06226)

In [ ]:
import tiktoken

enc = tiktoken.get_encoding("gpt2")
text = "Hello, LLM Training Handbook!"
tokens = enc.encode(text)
print(f"原文: {text}")
print(f"Tokens: {tokens}")
print(f"解码: {enc.decode(tokens)}")
print(f"词表大小: {enc.n_vocab}")

In [ ]:
def get_pair_counts(ids, counts=None):
    """统计相邻 token pair 的频率"""
    counts = {} if counts is None else counts
    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1
    return counts

def merge(ids, pair, idx):
    """将 ids 中所有 pair 替换为 idx"""
    new_ids = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i+1] == pair[1]:
            new_ids.append(idx)
            i += 2
        else:
            new_ids.append(ids[i])
            i += 1
    return new_ids

class MiniTokenizer:
    """Minimal BPE tokenizer for educational purposes."""
    def __init__(self, num_merges=100):
        self.num_merges = num_merges
        self.merges = {}
        self.vocab = {}

    def train(self, text):
        tokens = list(text.encode("utf-8"))
        vocab_size = 256 + self.num_merges
        for i in range(self.num_merges):
            counts = get_pair_counts(tokens)
            if not counts:
                break
            top_pair = max(counts, key=counts.get)
            idx = 256 + i
            tokens = merge(tokens, top_pair, idx)
            self.merges[top_pair] = idx
        self.vocab = {i: bytes([i]) for i in range(256)}
        for (p0, p1), idx in self.merges.items():
            self.vocab[idx] = self.vocab[p0] + self.vocab[p1]

    def encode(self, text):
        tokens = list(text.encode("utf-8"))
        while len(tokens) >= 2:
            counts = get_pair_counts(tokens)
            pair = min(counts, key=lambda p: self.merges.get(p, float("inf")))
            if pair not in self.merges:
                break
            tokens = merge(tokens, pair, self.merges[pair])
        return tokens

    def decode(self, ids):
        return b"".join(self.vocab[i] for i in ids).decode("utf-8", errors="replace")

# 训练并测试
with open("input.txt", "r") as f:
    text = f.read()
tokenizer = MiniTokenizer(num_merges=200)
tokenizer.train(text[:10000])
test = "Hello world"
encoded = tokenizer.encode(test)
print(f"Encoded: {encoded}")
print(f"Decoded: {tokenizer.decode(encoded)}")

分析 BPE 合并过程：打印前 20 次合并的 pair，可以看到 BPE 优先合并哪些字符组合。

你会观察到英文字母组合（如 'th', 'he', 'in'）最先被合并，因为它们在英文文本中出现频率最高。

In [ ]:
# 打印前 20 次 BPE 合并
print("前 20 次 BPE 合并:")
for i, ((p0, p1), idx) in enumerate(list(tokenizer.merges.items())[:20]):
    bytes_repr = tokenizer.vocab[idx]
    try:
        text_repr = bytes_repr.decode('utf-8')
    except:
        text_repr = bytes_repr.hex()
    print(f"  合并 {i+1}: ({p0}, {p1}) -> {idx} = '{text_repr}'")

In [ ]:
# Character-level tokenization (最简单的方式)
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: "".join([itos[i] for i in l])

print(f"词表大小: {vocab_size} (character-level)")
print(f"示例编码: {encode('hello')}")

In [ ]:
import torch

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

print(f"总 token 数: {len(data):,}")
print(f"训练集: {len(train_data):,} tokens")
print(f"验证集: {len(val_data):,} tokens")

def get_batch(split, block_size=128, batch_size=4):
    data_source = train_data if split == "train" else val_data
    ix = torch.randint(len(data_source) - block_size, (batch_size,))
    x = torch.stack([data_source[i:i+block_size] for i in ix])
    y = torch.stack([data_source[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch("train")
print(f"batch x shape: {xb.shape}, batch y shape: {yb.shape}")

## 练习

1. 对比三种 tokenizer 的压缩率：character-level、mini BPE、tiktoken GPT-2。对同一段文本，计算 token 数量比。
2. 修改 `MiniTokenizer` 的 `num_merges` 参数，观察合并次数对词表大小和压缩率的影响。
3. 思考：为什么中文需要更大的词表？BPE 对中文的处理有什么特殊之处？

## 延伸阅读

- [nanoGPT data/](https://github.com/karpathy/nanoGPT/tree/master/data) — 数据处理脚本参考
- [tiktoken](https://github.com/openai/tiktoken) — OpenAI 的高性能 BPE 实现
- [SentencePiece](https://github.com/google/sentencepiece) — Google 的多语言 tokenizer
- [BPE vs WordPiece vs Unigram](https://huggingface.co/docs/tokenizers/summary) — HuggingFace 的 tokenizer 对比